# U2T02: SimCSE


In [1]:
# Sustituye la celda antigua por esta. Conserva el PyTorch de Colab.
%pip install --upgrade --only-binary=:all: transformers==5.17.0 sentence-transformers==6.1.0 datasets==5.0.1 huggingface-hub==1.33.0 safetensors==0.8.0 numpy==2.2.6 pandas==2.2.3 scipy==1.15.2 matplotlib==3.10.1 scikit-learn==1.6.1 fsspec==2025.12.0
print("Instalación terminada. Reinicia la sesión antes de continuar.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 132.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 125.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: matplotlib
    Found existing install

Instalación terminada. Reinicia la sesión antes de continuar.


## Reinicia la sesión después de instalar
Después de instalar, reinicia la sesión y continúa.

In [1]:
import sys, gc, importlib.metadata as metadata
import numpy as np, pandas as pd, scipy, matplotlib, sklearn
import torch
from transformers import AutoModel, AutoTokenizer, BertConfig
from sentence_transformers import SentenceTransformer, models
from datasets import load_dataset
from huggingface_hub import HfApi
print("Python:",sys.version.split()[0])
for name in ['numpy','transformers','sentence-transformers','datasets','huggingface-hub','safetensors','torch']:
    print(name,metadata.version(name))
assert torch.cuda.is_available(), "Activa la GPU en Entorno de ejecución → Cambiar tipo de entorno."
print("GPU:",torch.cuda.get_device_name(0))
small=AutoModel.from_config(BertConfig(vocab_size=100,hidden_size=32,num_hidden_layers=1,num_attention_heads=4,intermediate_size=64)).cuda()
small.train()
ids=torch.ones((2,8),dtype=torch.long,device='cuda')
state_cpu=torch.get_rng_state();state_gpu=torch.cuda.get_rng_state_all()
with torch.no_grad():
    a=small(input_ids=ids,attention_mask=torch.ones_like(ids)).pooler_output
    torch.set_rng_state(state_cpu);torch.cuda.set_rng_state_all(state_gpu)
    b=small(input_ids=ids,attention_mask=torch.ones_like(ids)).pooler_output
assert a.shape==(2,32) and torch.isfinite(a).all()
assert torch.allclose(a,b,atol=1e-6), "No se reprodujo la máscara de dropout."
del small,a,b,ids;gc.collect();torch.cuda.empty_cache()
print("COMPROBACIÓN CORRECTA: continúa con Drive y el dataset.")


/tmp/ipykernel_13347/4239795854.py:5: DeprecationWarning: Importing from 'sentence_transformers.models' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.modules' instead.
  from sentence_transformers import SentenceTransformer, models


Python: 3.13.15
numpy 2.2.6
transformers 5.17.0
sentence-transformers 6.1.0
datasets 5.0.1
huggingface-hub 1.33.0
safetensors 0.8.0
torch 2.11.0+cu130
GPU: NVIDIA L4
COMPROBACIÓN CORRECTA: continúa con Drive y el dataset.


In [2]:
from google.colab import drive, files
drive.mount('/content/drive')
import os
os.environ['SIMCSE_OUTPUT']='/content/drive/MyDrive/U2T02_Diego'
DATA='/content/drive/MyDrive/U2T02_Diego/snli_train_100k.jsonl'
os.makedirs(os.path.dirname(DATA),exist_ok=True)
if not os.path.exists(DATA):
    uploaded=files.upload() # Select only snli_train_100k.jsonl
    if 'snli_train_100k.jsonl' not in uploaded: raise ValueError('Wrong filename')
    with open(DATA,'wb') as f:f.write(uploaded['snli_train_100k.jsonl'])


Mounted at /content/drive


In [3]:
"""U2T02 — Diego Jesus Loria Campos. Run from the accompanying notebook."""
import os, json, random, hashlib, platform, time, gc
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, set_seed, get_linear_schedule_with_warmup
from datasets import load_dataset
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer, models
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TOKENIZER = AutoTokenizer.from_pretrained('bert-base-uncased')
ROOT = Path(os.environ.get('SIMCSE_OUTPUT', '/content/drive/MyDrive/U2T02_Diego'))
ROOT.mkdir(parents=True, exist_ok=True)
MAX_LENGTH = 64

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False))

def prepare(path):
    rows = [json.loads(s) for s in Path(path).read_text().splitlines() if s.strip()]
    sentences = sorted({s for r in rows for s in (r['premise'], r['hypothesis'])})
    contradictions = defaultdict(list)
    for r in rows:
        assert r['label'] in (0,1,2)
        if r['label']==2: contradictions[r['premise']].append(r['hypothesis'])
    pairs = [(r['premise'],r['hypothesis'],sorted(set(contradictions[r['premise']]))) for r in rows if r['label']==0]
    stats = dict(records=len(rows), unique_sentences=len(sentences), entailments=len(pairs),
                 with_hard_negative=sum(bool(p[2]) for p in pairs), sha256=hashlib.sha256(Path(path).read_bytes()).hexdigest())
    assert (len(rows),len(sentences),len(pairs)) == (100000,165529,33351), stats
    write_json(ROOT/'data_stats.json',stats)
    return sentences,pairs,stats

class Encoder(nn.Module):
    def __init__(self, mode, path='bert-base-uncased', dropout=.1):
        super().__init__()
        self.mode=mode
        self.bert=AutoModel.from_pretrained(path, hidden_dropout_prob=dropout, attention_probs_dropout_prob=dropout)
    def forward(self,tokens,training=False):
        o=self.bert(**tokens)
        # SimCSE: CLS + original BERT dense/tanh head during training.
        # Unsupervised evaluation removes that head; supervised retains it.
        return o.pooler_output if training or self.mode=='sup' else o.last_hidden_state[:,0]

@torch.no_grad()
def embeddings(model, sentences, batch=64):
    if isinstance(model,SentenceTransformer):
        return model.encode(sentences,batch_size=batch,normalize_embeddings=True,show_progress_bar=False)
    model.eval(); result=[]
    for i in range(0,len(sentences),batch):
        t=TOKENIZER(sentences[i:i+batch],padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt').to(DEVICE)
        if model.mode=='mean':
            h=model.bert(**t).last_hidden_state; m=t['attention_mask'].unsqueeze(-1)
            z=(h*m).sum(1)/m.sum(1)
        else: z=model(t)
        result.append(F.normalize(z,dim=-1).cpu().numpy())
    return np.concatenate(result)

def geometry(a,b,scores):
    # Positive STS pairs: human score >= 4/5, identical protocol across models.
    pos=np.asarray(scores)>=.8
    alignment=float(np.square(a[pos]-b[pos]).sum(1).mean())
    z=np.concatenate([a,b]); rng=np.random.default_rng(123)
    i=rng.integers(len(z),size=20000); j=rng.integers(len(z)-1,size=20000); j+=j>=i
    uniformity=float(np.log(np.exp(-2*np.square(z[i]-z[j]).sum(1)).mean()))
    return alignment,uniformity

def evaluate(model, data, name=None, split='dev'):
    a=embeddings(model,list(data['sentence1'])); b=embeddings(model,list(data['sentence2']))
    scores=np.asarray(data['score']); cosine=(a*b).sum(1)
    al,un=geometry(a,b,scores)
    result=dict(spearman=float(spearmanr(cosine,scores).statistic*100),alignment=al,uniformity=un)
    if name:
        dest=ROOT/name; dest.mkdir(exist_ok=True)
        np.savez_compressed(dest/f'{split}_predictions.npz',a=a,b=b,score=scores,cosine=cosine)
        write_json(dest/f'{split}_metrics.json',result)
    return result

def rng_state(): return (torch.get_rng_state(),torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)
def restore_rng(state):
    torch.set_rng_state(state[0])
    if state[1] is not None: torch.cuda.set_rng_state_all(state[1])

def train(config,sentences,pairs,dev):
    name=config['name']; dest=ROOT/name
    if (dest/'completed.json').exists(): return json.loads((dest/'completed.json').read_text())
    dest.mkdir(exist_ok=True); set_seed(config['seed'])
    model=Encoder(config['mode'],dropout=config['dropout']).to(DEVICE)
    data=sentences if config['mode']=='unsup' else pairs
    generator=torch.Generator().manual_seed(config['seed'])
    loader=DataLoader(data,batch_size=config['batch_size'],shuffle=True,generator=generator,collate_fn=lambda x:x)
    opt=torch.optim.AdamW(model.parameters(),lr=config['lr'],weight_decay=0.)
    total=len(loader)*config['epochs']
    sched=get_linear_schedule_with_warmup(opt,int(total*.1),total)
    best=-float('inf'); step=0; history=[]; started=time.time()
    write_json(dest/'config.json',{**config,'hardware':torch.cuda.get_device_name(0) if DEVICE=='cuda' else platform.processor(), 'torch':torch.__version__,'max_length':MAX_LENGTH})
    for epoch in range(config['epochs']):
        for items in loader:
            model.train(); opt.zero_grad(set_to_none=True)
            left=items if config['mode']=='unsup' else [x[0] for x in items]
            right=items if config['mode']=='unsup' else [x[1] for x in items]
            # Joint padding means RNG replay reproduces masks at all dropout layers.
            toks=TOKENIZER(left+right,padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt').to(DEVICE)
            n=len(items); t1={k:v[:n] for k,v in toks.items()}; t2={k:v[n:] for k,v in toks.items()}
            before=rng_state(); z1=F.normalize(model(t1,True),dim=-1)
            if config['same_mask']: restore_rng(before)
            z2=F.normalize(model(t2,True),dim=-1)
            logits=z1@z2.T/config['temperature']
            if config['mode']=='sup' and config['hard_negatives']:
                # Only actual contradictions are used, never neutral or invented fallback negatives.
                negatives=[x[2][epoch%len(x[2])] for x in items if x[2]]
                if negatives:
                    tn=TOKENIZER(negatives,padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt').to(DEVICE)
                    zn=F.normalize(model(tn,True),dim=-1)
                    logits=torch.cat([logits,z1@zn.T/config['temperature']],dim=1)
            loss=F.cross_entropy(logits,torch.arange(n,device=DEVICE))
            if not torch.isfinite(loss): raise RuntimeError(f'Non-finite loss: {name} step {step}')
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.); opt.step(); sched.step(); step+=1
            if step%100==0: print(name,step,'loss',round(loss.item(),4),flush=True)
            if step%config['eval_steps']==0 or step==total:
                metrics=evaluate(model,dev)
                history.append(dict(step=step,epoch=epoch+1,loss=float(loss),**metrics))
                pd.DataFrame(history).to_csv(dest/'training_log.csv',index=False)
                if metrics['spearman']>best:
                    best=metrics['spearman']; model.bert.save_pretrained(dest/'best'); TOKENIZER.save_pretrained(dest/'best')
                print(name,'dev',metrics['spearman'],'best',best,flush=True)
    del model; gc.collect(); torch.cuda.empty_cache()
    model=Encoder(config['mode'],str(dest/'best')).to(DEVICE)
    result={**config,'dev':evaluate(model,dev,name),'seconds':time.time()-started}
    write_json(dest/'completed.json',result)
    with (ROOT/'runs.jsonl').open('a') as f:f.write(json.dumps(result)+'\n')
    del model;gc.collect();torch.cuda.empty_cache()
    return result

def baseline_dev(dev):
    bert=Encoder('mean').to(DEVICE)
    first=evaluate(bert,dev,'bert_mean'); del bert;gc.collect();torch.cuda.empty_cache()
    sbert=SentenceTransformer('sentence-transformers/bert-base-nli-mean-tokens',device=DEVICE)
    second=evaluate(sbert,dev,'sbert_2019'); del sbert;gc.collect();torch.cuda.empty_cache()
    write_json(ROOT/'baseline_dev.json',{'bert_mean':first,'sbert_2019':second})
    print('Dev ×100:',first,second)
    # Stop before training when evaluation is far from assignment references.
    if abs(first['spearman']-59.31)>3 or abs(second['spearman']-80.77)>3:
        raise RuntimeError('Baseline mismatch >3 points. Check pooling, truncation, dataset and checkpoint before training.')
    return first,second

def final_benchmark(configs,dev,test):
    marker=ROOT/'final_test_started.json'
    if marker.exists(): raise RuntimeError('Test phase already started. Use saved results; do not select or retune using test.')
    write_json(marker,{'configs':configs,'time':time.time()})
    rows=[]
    for name in ['bert_mean','sbert_2019']+[c['name'] for c in configs]:
        if name=='bert_mean': model=Encoder('mean').to(DEVICE)
        elif name=='sbert_2019': model=SentenceTransformer('sentence-transformers/bert-base-nli-mean-tokens',device=DEVICE)
        else:
            config=next(c for c in configs if c['name']==name)
            model=Encoder(config['mode'],str(ROOT/name/'best')).to(DEVICE)
        dv=json.loads((ROOT/name/'dev_metrics.json').read_text())
        tv=evaluate(model,test,name,'test')
        rows.append(dict(model=name,dev_spearman=dv['spearman'],test_spearman=tv['spearman'],alignment=tv['alignment'],uniformity=tv['uniformity']))
        pd.DataFrame(rows).to_csv(ROOT/'benchmark.csv',index=False)
        del model;gc.collect();torch.cuda.empty_cache()
    return pd.DataFrame(rows)

def analyze(configs,dev):
    names=['bert_mean','sbert_2019']+[c['name'] for c in configs]
    for name in names:
        dest=ROOT/name; d=np.load(dest/'dev_predictions.npz')
        bins=np.minimum((d['score']*5).astype(int),4)
        plt.figure(figsize=(7,4));plt.boxplot([d['cosine'][bins==i] for i in range(5)],tick_labels=['0–1','1–2','2–3','3–4','4–5'])
        plt.xlabel('Human score (0–5)');plt.ylabel('Cosine similarity');plt.title(name);plt.tight_layout();plt.savefig(dest/'similarity.png',dpi=160);plt.close()
        # Held-out dev retrieval, excluding exact self matches. Review examples manually.
        corpus=list(dict.fromkeys(list(dev['sentence1'])+list(dev['sentence2'])))
        embs={s:z for s,z in zip(dev['sentence1'],d['a'])};embs.update({s:z for s,z in zip(dev['sentence2'],d['b'])})
        matrix=np.stack([embs[s] for s in corpus]); examples=[]
        # Include the highest model similarity among low-rated pairs as a failure candidate.
        low=np.flatnonzero(d['score']<=.2); fail=int(low[np.argmax(d['cosine'][low])])
        indices=[0,10,100,fail]
        for idx in indices:
            query=dev[idx]['sentence1'];sim=matrix@embs[query];sim[np.array(corpus)==query]=-np.inf
            top=np.argsort(-sim)[:5]
            examples.append({'query':query,'paired_sentence':dev[idx]['sentence2'],'human_score':float(dev[idx]['score']*5),'pair_cosine':float(d['cosine'][idx]),'failure_candidate':idx==fail,'neighbors':[{'sentence':corpus[k],'cosine':float(sim[k])} for k in top]})
        write_json(dest/'retrievals.json',examples)
    b=pd.read_csv(ROOT/'benchmark.csv');plt.figure(figsize=(7,5))
    for r in b.itertuples():plt.scatter(r.uniformity,r.alignment);plt.annotate(r.model,(r.uniformity,r.alignment),fontsize=8)
    plt.xlabel('Uniformity (lower = more uniform)');plt.ylabel('Alignment (lower = closer positives)');plt.tight_layout();plt.savefig(ROOT/'geometry.png',dpi=160);plt.close()
    deltas=[]
    for mode,a,bname in [('unsup','unsup_s42','unsup_same_mask_s42'),('sup','sup_hard_s42','sup_nohard_s42')]:
        table=b.set_index('model');deltas.append({'mode':mode,'baseline':a,'ablation':bname,'dev_delta':float(table.loc[bname,'dev_spearman']-table.loc[a,'dev_spearman']),'test_delta':float(table.loc[bname,'test_spearman']-table.loc[a,'test_spearman'])})
    write_json(ROOT/'ablation_deltas.json',deltas)

def export_publish(config,repo_id,dev,test):
    source=ROOT/config['name']/'best'
    transformer=models.Transformer(str(source),max_seq_length=MAX_LENGTH)
    pool=models.Pooling(transformer.get_word_embedding_dimension(),pooling_mode='cls')
    modules=[transformer,pool]
    if config['mode']=='sup':
        dense=models.Dense(pool.get_sentence_embedding_dimension(),pool.get_sentence_embedding_dimension(),activation_function=nn.Tanh())
        dense.linear.load_state_dict(transformer.auto_model.pooler.dense.state_dict())
        modules.append(dense)
    modules.append(models.Normalize()); st=SentenceTransformer(modules=modules,device=DEVICE)
    local=ROOT/config['name']/'sentence_transformers';st.save(str(local))
    original=json.loads((ROOT/config['name']/'test_metrics.json').read_text())
    card=f'''---\nlanguage: en\nlibrary_name: sentence-transformers\ntags: [simcse, sentence-similarity]\n---\n# U2T02 SimCSE\nAuthor: Diego Jesus Loria Campos; add remaining team members before publishing.\n\nTraining: provided SNLI train-only 100k subset; 165529 unique sentences / 33351 entailment pairs; 9488 pairs have contradictions. No STS-B training.\n\nRecipe: `{json.dumps(config)}`. Max length: {MAX_LENGTH}; best checkpoint selected using STS-B dev.\n\nTest results: `{json.dumps(original)}`. Spearman ×100; normalized cosine, no regressor. Alignment: score >=0.8 pairs, alpha=2; uniformity: t=2, 20000 sampled pairs.\n\nLimitations: English only, short sentences, limited NLI data, domain bias, no claim of medical or factual reliability. Semantic similarity is not factual entailment.\n'''
    (local/'README.md').write_text(card)
    # Verify local export on DEV first. Only required Hub reload repeats final test.
    expected=json.loads((ROOT/config['name']/'dev_metrics.json').read_text())['spearman']
    measured=evaluate(st,dev)['spearman']
    assert abs(measured-expected)<.05,(measured,expected)
    from huggingface_hub import HfApi
    api=HfApi(); api.create_repo(repo_id,exist_ok=True)
    api.upload_folder(repo_id=repo_id,folder_path=str(local))
    reloaded=SentenceTransformer(repo_id,device=DEVICE)
    verification=evaluate(reloaded,test)
    assert abs(verification['spearman']-original['spearman'])<.05,(verification,original)
    write_json(ROOT/'hub_verification.json',{'repo_id':repo_id,'expected':original,'reloaded':verification})
    return verification


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [4]:
assert DEVICE=='cuda', 'Activate Colab: Runtime > Change runtime type > GPU'
import subprocess
(ROOT/'requirements_actual.txt').write_text(subprocess.check_output(['pip','freeze'],text=True))
sentences,pairs,stats=prepare(DATA)
print(stats)
stsb=load_dataset('sentence-transformers/stsb')
dev=stsb['validation']
assert len(dev)==1500 and len(stsb['test'])==1379
baseline_dev(dev) # Must pass BEFORE training.


{'records': 100000, 'unique_sentences': 165529, 'entailments': 33351, 'with_hard_negative': 9488, 'sha256': '0927a0b1d2a447b9236fd9699d6a37a4a3542c5def178d701de1c38827997f88'}


README.md:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  471kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  142kB            

...

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dev ×100: {'spearman': 59.31201141819942, 'alignment': 0.1948346644639969, 'uniformity': -1.6318234205245972} {'spearman': 80.77414503337994, 'alignment': 0.19287589192390442, 'uniformity': -3.053150177001953}


({'spearman': 59.31201141819942,
  'alignment': 0.1948346644639969,
  'uniformity': -1.6318234205245972},
 {'spearman': 80.77414503337994,
  'alignment': 0.19287589192390442,
  'uniformity': -3.053150177001953})

In [5]:
SEEDS=[42,43,44] # [42] for first pass; three seeds are needed to estimate run-to-run noise.
configs=[]
for seed in SEEDS:
    common=dict(seed=seed,batch_size=64,temperature=.05,epochs=1,dropout=.1,eval_steps=250)
    configs.extend([
        dict(common,name=f'unsup_s{seed}',mode='unsup',lr=3e-5,same_mask=False,hard_negatives=False),
        dict(common,name=f'unsup_same_mask_s{seed}',mode='unsup',lr=3e-5,same_mask=True,hard_negatives=False),
        dict(common,name=f'sup_hard_s{seed}',mode='sup',lr=5e-5,same_mask=False,hard_negatives=True),
        dict(common,name=f'sup_nohard_s{seed}',mode='sup',lr=5e-5,same_mask=False,hard_negatives=False),
    ])
# Change batch size for ALL matched experiments before starting if GPU memory is insufficient.
for config in configs: train(config,sentences,pairs,dev)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


unsup_s42 100 loss 0.9815
unsup_s42 200 loss 0.0334


/tmp/ipykernel_13347/2621461395.py:128: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  history.append(dict(step=step,epoch=epoch+1,loss=float(loss),**metrics))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

sup_nohard_s44 dev 75.97841121828233 best 75.97841121828233


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

sup_nohard_s44 dev 76.0516066910251 best 76.0516066910251


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Freeze the experiment before using test
Evaluación final
Selecciona los checkpoints usando dev. Evalúa test una sola vez, sin usarlo para ajustar el modelo. La verificación después de descargarlo del Hub es una comprobación adicional.


In [6]:
test=stsb['test']
benchmark=final_benchmark(configs,dev,test)
display(benchmark)
analyze(configs,dev)
# Across-seed noise and paired ablation differences (descriptive; n=3 is small).
noise=benchmark.copy()
noise['variant']=noise['model'].str.replace(r'_s\d+$','',regex=True)
display(noise.groupby('variant')[['dev_spearman','test_spearman']].agg(['mean','std']))
noise.groupby('variant')[['dev_spearman','test_spearman']].agg(['mean','std']).to_csv(ROOT/'seed_noise.csv')
for prefix,abl in [('unsup','unsup_same_mask'),('sup_hard','sup_nohard')]:
    t=benchmark.set_index('model'); diffs=[]
    for seed in SEEDS:
        diffs.append({k:float(t.loc[f'{abl}_s{seed}',k]-t.loc[f'{prefix}_s{seed}',k]) for k in ['dev_spearman','test_spearman']})
    pd.DataFrame(diffs).to_csv(ROOT/f'{prefix}_paired_deltas.csv',index=False)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,model,dev_spearman,test_spearman,alignment,uniformity
0,bert_mean,59.312011,47.289437,0.215541,-1.613616
1,sbert_2019,80.774145,76.985237,0.179769,-3.050959
2,unsup_s42,77.173179,69.841386,0.227366,-2.376544
3,unsup_same_mask_s42,62.749394,53.137850,0.404503,-2.662651
4,sup_hard_s42,76.738805,73.626239,0.108420,-2.216590
5,sup_nohard_s42,75.990352,73.652746,0.105229,-2.137752
6,unsup_s43,77.705048,69.762270,0.225241,-2.475319
7,unsup_same_mask_s43,69.328610,59.294265,0.397411,-2.803131
8,sup_hard_s43,76.311114,73.951980,0.121841,-2.257201
9,sup_nohard_s43,76.513959,72.862804,0.109380,-2.119101


dev_spearman           test_spearman          
                        mean       std          mean       std
variant                                                       
bert_mean          59.312011       NaN     47.289437       NaN
sbert_2019         80.774145       NaN     76.985237       NaN
sup_hard           76.362614  0.353268     73.752528  0.174761
sup_nohard         76.185306  0.286265     73.030547  0.557585
unsup              76.758868  1.207860     68.562841  2.146354
unsup_same_mask    67.114049  3.780039     57.374725  3.674629

## Publish


In [8]:
REPO_ID = 'Perry-DLC/upy-u2t02-simcse'

candidates = [
    c for c in configs
    if c['name'] in ['unsup_s42', 'sup_hard_s42']
]

best = max(
    candidates,
    key=lambda c: json.loads(
        (ROOT / c['name'] / 'dev_metrics.json').read_text()
    )['spearman']
)

print('Selected using dev:', best['name'])
verification = export_publish(best, REPO_ID, dev, test)
print('Verificación:', verification)
print(f'Modelo publicado: https://huggingface.co/{REPO_ID}')

Selected using dev: unsup_s42


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_13347/2621461395.py:201: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  pool=models.Pooling(transformer.get_word_embedding_dimension(),pooling_mode='cls')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/89.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

Verificación: {'spearman': 69.841380171436, 'alignment': 0.2273658961057663, 'uniformity': -2.3765439987182617}
Modelo publicado: https://huggingface.co/Perry-DLC/upy-u2t02-simcse


## Descargar métricas, logs y gráficas

Esta descarga contiene las evidencias; los pesos del modelo publicado están en Hugging Face.

In [1]:
from google.colab import drive, files
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

drive.mount('/content/drive')

carpeta = Path('/content/drive/MyDrive/U2T02_Diego')
archivo = Path('/content/U2T02_resultados_reporte.zip')
extensiones = {'.json', '.csv', '.png', '.txt', '.md'}

if not carpeta.is_dir():
    raise FileNotFoundError(
        f"No se encontró la carpeta: {carpeta}. "
        "Comprueba que conectaste la cuenta de Drive usada ayer."
    )

rutas = [
    ruta for ruta in carpeta.rglob('*')
    if ruta.is_file() and ruta.suffix.lower() in extensiones
]

if not rutas:
    raise RuntimeError("No se encontraron archivos de resultados en la carpeta.")

with ZipFile(archivo, 'w', compression=ZIP_DEFLATED) as zip_file:
    for ruta in rutas:
        zip_file.write(ruta, ruta.relative_to(carpeta))

# Guardar también el ZIP pequeño en Drive.
import shutil
copia_drive = carpeta / archivo.name
shutil.copy2(archivo, copia_drive)

print(f"Archivos incluidos: {len(rutas)}")
print(f"Tamaño: {archivo.stat().st_size / (1024**2):.2f} MB")
print(f"Copia guardada en Drive: {copia_drive}")

files.download(str(archivo))

Mounted at /content/drive
Archivos incluidos: 148
Tamaño: 3.08 MB
Copia guardada en Drive: /content/drive/MyDrive/U2T02_Diego/U2T02_resultados_reporte.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>